# IA3 Mini Project: Patient Health Indicators and Length of Hospital Stay: Data Cleaning, EDA and Hypothesis Testing

**Course:** CES 2103 Data Analytics, MIT Manipal

**Problem statement:** To clean a raw dataset of 30,000 patient records
and find out how patients with different medical conditions differ in length
of stay, HbA1c and smoking, using hypothesis tests to check whether the
differences are real.

**Description:** Cleaning, exploring and testing a patient health dataset
(`hospital_dataset.csv`). The notebook has four parts: preprocessing,
exploratory data analysis (EDA), confirmatory data analysis (CDA) using
z-tests and t-tests, and conclusions.

**Team number:** 4

| Member | Name | Registration number |
|---|---|---|
| 1 | Aryan Basireddy | 251090051178 |
| 2 | Kallem Rithvik Reddy | 251090050888 |
| 3 | Dinker Varma Siruvuri | 251090050182 |
| 4 | Nadha Pranav Visintaravula | 251090051098 |
| 5 | Mandadi Manas Chandra | 251090052450 |

**Date:** 26th October 2026

In [ ]:
# Imports are kept together at the top of the notebook.
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Constants are written in upper case.
DATA_FILE = "dirty_v3_path.csv"
ALPHA = 0.05

# All figures are saved in this folder so they can be used in the report.
os.makedirs("figs", exist_ok=True)
sns.set_theme(style="whitegrid")

# This dictionary collects the numbers that are used in the report.
results = {}

## 1. Loading and inspecting the data

In [ ]:
# Read the csv file and look at its size.
raw_df = pd.read_csv(DATA_FILE)
print("Rows and columns:", raw_df.shape)
results["raw_rows"] = int(raw_df.shape[0])
results["raw_cols"] = int(raw_df.shape[1])
raw_df.head(10)

In [ ]:
# Data types and non-null counts of every column.
raw_df.info()

In [ ]:
# Count missing values and duplicate rows.
missing_counts = raw_df.isnull().sum()
print(missing_counts[missing_counts > 0])

complete_rows = len(raw_df.dropna())
duplicate_rows = int(raw_df.duplicated().sum())
print("Rows with no missing value at all:", complete_rows)
print("Duplicate rows:", duplicate_rows)

only_missing = missing_counts[missing_counts > 0]
results["missing_per_col"] = {
    col: int(count) for col, count in only_missing.items()
}
results["complete_rows"] = int(complete_rows)
results["duplicates"] = duplicate_rows

## 2. Data preprocessing

The steps are: drop useless columns, handle missing values, handle noisy
(impossible) values, check outliers, and create bins.

### 2.1 Dropping columns that carry no information

In [ ]:
# random_notes only has four filler values and noise_col looks like noise.
df = raw_df.copy()
print(df["random_notes"].value_counts())

number_cols = df.select_dtypes(include="number").columns.drop("noise_col")
noise_corr = df[number_cols].corrwith(df["noise_col"]).abs().max()
print("Largest correlation of noise_col:", round(noise_corr, 3))

df = df.drop(columns=["random_notes", "noise_col"])
results["noise_corr"] = float(noise_corr)
results["notes_counts"] = {
    key: int(value)
    for key, value in raw_df["random_notes"].value_counts().items()
}

### 2.2 Missing values

Medical Condition is the column we group by in the whole project, so rows
where it is missing are removed. Age, Glucose and Blood Pressure are filled
with the arithmetic mean of the column. Gender is filled with "Unknown"
because we cannot guess it.

In [ ]:
# Remove rows that have no Medical Condition.
rows_before = len(df)
df = df.dropna(subset=["Medical Condition"]).reset_index(drop=True)
results["dropped_condition"] = int(rows_before - len(df))
results["rows_after_drop"] = int(len(df))
print("Rows removed:", rows_before - len(df), "Rows left:", len(df))

# Summary of the columns that will be filled, before filling.
mean_cols = ["Age", "Glucose", "Blood Pressure"]
stat_rows = ["mean", "std", "min", "25%", "50%", "75%", "max"]
summary_before = df[mean_cols].describe().loc[stat_rows].T.round(2)
summary_before

In [ ]:
# Fill the missing values with the mean of each column.
results["missing_filled"] = {
    col: int(df[col].isnull().sum()) for col in mean_cols
}

# Remember which glucose values were really measured. Later we use only
# these rows for the scatter plot, because filled values would all sit on
# one vertical line.
df["Glucose Missing"] = df["Glucose"].isnull()

for col in mean_cols:
    df[col] = df[col].fillna(df[col].mean())

# Age is a whole number, so it is rounded after filling.
df["Age"] = df["Age"].round().astype(int)

# Gender cannot be guessed, so it gets its own label.
results["gender_unknown"] = int(df["Gender"].isnull().sum())
df["Gender"] = df["Gender"].fillna("Unknown")

summary_after = df[mean_cols].describe().loc[stat_rows].T.round(2)
results["summary_before"] = summary_before.to_dict("index")
results["summary_after"] = summary_after.to_dict("index")
summary_after

**Was mean filling effective?** The mean stays almost the same, which is
good, but the standard deviation becomes smaller because thousands of rows
now have exactly the same value. So the method keeps the centre of the data
but makes it look a little less spread out than it really is.

### 2.3 Noisy values

Some values cannot exist in real life: oxygen saturation above 100%,
negative physical activity, diet or stress scores, and triglycerides of zero
or below. These are replaced with the median of the valid values of that
column.

In [ ]:
# Keep a copy of oxygen saturation to compare before and after.
spo2_before = df["Oxygen Saturation"].copy()
noisy_counts = {}

# Oxygen saturation cannot be more than 100 percent.
bad_rows = df["Oxygen Saturation"] > 100
valid_median = df.loc[~bad_rows, "Oxygen Saturation"].median()
df.loc[bad_rows, "Oxygen Saturation"] = valid_median
noisy_counts["Oxygen Saturation"] = int(bad_rows.sum())

# These three scores start at zero, so negative values are noise.
for col in ["Physical Activity", "Diet Score", "Stress Level"]:
    bad_rows = df[col] < 0
    valid_median = df.loc[~bad_rows, col].median()
    df.loc[bad_rows, col] = valid_median
    noisy_counts[col] = int(bad_rows.sum())

# Triglycerides cannot be zero or negative.
bad_rows = df["Triglycerides"] <= 0
valid_median = df.loc[~bad_rows, "Triglycerides"].median()
df.loc[bad_rows, "Triglycerides"] = valid_median
noisy_counts["Triglycerides"] = int(bad_rows.sum())

results["noisy_counts"] = noisy_counts
print(noisy_counts)

In [ ]:
# Box plots of oxygen saturation before and after treating the noise.
fig, axes = plt.subplots(1, 2, figsize=(7, 3), sharey=True)
sns.boxplot(y=spo2_before, ax=axes[0], color="lightcoral")
axes[0].set_title("Before cleaning")
axes[0].set_ylabel("Oxygen saturation (%)")
sns.boxplot(y=df["Oxygen Saturation"], ax=axes[1], color="steelblue")
axes[1].set_title("After cleaning")
axes[1].set_ylabel("")
plt.tight_layout()
plt.savefig("figs/fig0_spo2_cleaning.png", dpi=150)
plt.show()

results["spo2_median_before"] = float(spo2_before.median())
results["spo2_max_before"] = float(spo2_before.max())
results["spo2_max_after"] = float(df["Oxygen Saturation"].max())

### 2.4 Outliers

Outliers are found with the IQR rule: a value is an outlier if it is below
Q1 - 1.5 x IQR or above Q3 + 1.5 x IQR.

In [ ]:
# Count the IQR outliers in the main numeric columns.
iqr_counts = {}
check_cols = ["Glucose", "Blood Pressure", "BMI", "Cholesterol",
              "Triglycerides", "HbA1c", "LengthOfStay"]

for col in check_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low_limit = q1 - 1.5 * iqr
    high_limit = q3 + 1.5 * iqr
    outliers = df[(df[col] < low_limit) | (df[col] > high_limit)]
    iqr_counts[col] = int(len(outliers))

results["iqr_counts"] = iqr_counts
print(iqr_counts)

These outliers are kept. High glucose and HbA1c belong to diabetic patients
and long stays belong to cancer patients, so they are real values and
removing them would remove the groups we want to compare.

### 2.5 Binning

Age and BMI are converted into groups with `pd.cut`.

In [ ]:
# Age groups and BMI categories.
df["Age Group"] = pd.cut(
    df["Age"], bins=[0, 17, 39, 59, 120],
    labels=["Under 18", "18-39", "40-59", "60+"])

df["BMI Category"] = pd.cut(
    df["BMI"], bins=[0, 18.5, 25, 30, 100], right=False,
    labels=["Underweight", "Normal", "Overweight", "Obese"])

print(df["Age Group"].value_counts().sort_index())
print(df["BMI Category"].value_counts().sort_index())

results["age_group_counts"] = {
    key: int(value)
    for key, value in df["Age Group"].value_counts().sort_index().items()
}
results["bmi_cat_counts"] = {
    key: int(value)
    for key, value in df["BMI Category"].value_counts().sort_index().items()
}

In [ ]:
# Final check and saving the cleaned data.
print("Missing values left:", int(df.isnull().sum().sum()))
print("Cleaned shape:", df.shape)
results["clean_rows"] = int(df.shape[0])
results["clean_cols"] = int(df.shape[1])
df.to_csv("cleaned_data.csv", index=False)

## 3. Exploratory data analysis (EDA)

### 3.1 Summary statistics

In [ ]:
# Mean, median, spread, minimum and maximum of the main numeric columns.
summary_cols = ["Age", "Glucose", "Blood Pressure", "BMI",
                "Oxygen Saturation", "LengthOfStay", "Cholesterol",
                "Triglycerides", "HbA1c"]
summary_table = df[summary_cols].describe().round(2)
results["describe"] = summary_table.to_dict()
summary_table

In [ ]:
# Categorical columns.
print(df["Medical Condition"].value_counts())
print(df["Gender"].value_counts())

rates = df[["Smoking", "Alcohol", "Family History"]].mean()
print(rates.round(3))

results["condition_counts"] = {
    key: int(value)
    for key, value in df["Medical Condition"].value_counts().items()
}
results["gender_counts"] = {
    key: int(value) for key, value in df["Gender"].value_counts().items()
}
results["overall_rates"] = {key: float(v) for key, v in rates.items()}
results["skew"] = {col: float(df[col].skew()) for col in summary_cols}

### 3.2 Distributions

In [ ]:
# Histograms with a density curve for six important columns.
hist_cols = ["Age", "Glucose", "BMI", "HbA1c", "LengthOfStay",
             "Oxygen Saturation"]

fig, axes = plt.subplots(2, 3, figsize=(9, 5))
for ax, col in zip(axes.ravel(), hist_cols):
    sns.histplot(df[col], bins=30, kde=True, ax=ax, color="steelblue")
    ax.set_title(col)
    ax.set_xlabel("")
    ax.set_ylabel("")
plt.tight_layout()
plt.savefig("figs/fig1_histograms.png", dpi=150)
plt.show()

### 3.3 Patients and lifestyle by condition

In [ ]:
# Number of patients per condition, and smoking and alcohol percentages.
count_order = df["Medical Condition"].value_counts().index
life_rates = df.groupby("Medical Condition")[["Smoking", "Alcohol"]].mean()
life_rates = life_rates.loc[count_order] * 100

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
sns.countplot(data=df, x="Medical Condition", order=count_order,
              color="steelblue", ax=axes[0])
axes[0].set_title("Patients per condition")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=40)

life_rates.plot(kind="bar", ax=axes[1], color=["darkorange", "seagreen"])
axes[1].set_title("Smoking and alcohol use by condition")
axes[1].set_xlabel("")
axes[1].set_ylabel("% of patients")
axes[1].tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.savefig("figs/fig2_categorical.png", dpi=150)
plt.show()

results["alcohol_range"] = [float(life_rates["Alcohol"].min()),
                            float(life_rates["Alcohol"].max())]

### 3.4 Comparing conditions

In [ ]:
# Box plots of length of stay and HbA1c for each condition.
mean_order = (df.groupby("Medical Condition")["LengthOfStay"]
                .mean().sort_values().index)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
sns.boxplot(data=df, x="Medical Condition", y="LengthOfStay",
            order=mean_order, color="lightsteelblue", ax=axes[0])
axes[0].set_title("Length of stay by condition")
axes[0].set_ylabel("Days")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=40)

sns.boxplot(data=df, x="Medical Condition", y="HbA1c",
            order=mean_order, color="sandybrown", ax=axes[1])
axes[1].set_title("HbA1c by condition")
axes[1].set_ylabel("HbA1c (%)")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=40)
plt.tight_layout()
plt.savefig("figs/fig3_boxplots.png", dpi=150)
plt.show()

In [ ]:
# Group summary: one row per condition.
group_table = df.groupby("Medical Condition").agg(
    patients=("Age", "count"),
    mean_age=("Age", "mean"),
    mean_glucose=("Glucose", "mean"),
    mean_hba1c=("HbA1c", "mean"),
    mean_stay=("LengthOfStay", "mean"),
    median_stay=("LengthOfStay", "median"),
    smoking_rate=("Smoking", "mean"),
).round(2)
group_table = group_table.loc[mean_order]
results["group_table"] = group_table.to_dict("index")
group_table

In [ ]:
# Cross-tabulation: share of smokers and non-smokers in each condition.
smoke_table = pd.crosstab(df["Medical Condition"], df["Smoking"],
                          normalize="index").round(3)
smoke_table

In [ ]:
# Pivot table: average length of stay by condition and age group.
pivot_stay = pd.pivot_table(df, values="LengthOfStay",
                            index="Medical Condition", columns="Age Group",
                            aggfunc="mean", observed=False).round(2)
results["pivot_stay"] = pivot_stay.to_dict("index")
pivot_stay

In [ ]:
# Average HbA1c for each BMI category.
bmi_hba1c = df.groupby("BMI Category", observed=False)["HbA1c"].mean()
results["bmi_hba1c"] = {key: float(v) for key, v in bmi_hba1c.items()}
bmi_hba1c.round(2)

### 3.5 Relationships between columns

In [ ]:
# Correlation heatmap (left) and glucose vs HbA1c scatter plot (right).
corr_cols = ["Age", "Glucose", "Blood Pressure", "BMI", "Oxygen Saturation",
             "LengthOfStay", "Cholesterol", "Triglycerides", "HbA1c",
             "Physical Activity", "Diet Score", "Stress Level",
             "Sleep Hours", "Smoking", "Alcohol", "Family History"]
corr = df[corr_cols].corr()

# Only rows where glucose was really measured are used for the scatter plot.
measured = df[~df["Glucose Missing"]]
sample = measured.sample(3000, random_state=1)

fig, axes = plt.subplots(1, 2, figsize=(11, 5),
                         gridspec_kw={"width_ratios": [1.4, 1]})
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            annot_kws={"size": 5.5}, cbar_kws={"shrink": 0.7}, ax=axes[0])
axes[0].set_title("Correlation matrix")

sns.scatterplot(data=sample, x="Glucose", y="HbA1c",
                hue="Medical Condition", s=8, alpha=0.6, ax=axes[1])
axes[1].set_title("Glucose vs HbA1c (3,000 measured rows)")
axes[1].legend(fontsize=6, markerscale=1.5)
plt.tight_layout()
plt.savefig("figs/fig4_corr_scatter.png", dpi=150)
plt.show()

# Straight line fit using the measured rows.
slope, intercept = np.polyfit(measured["Glucose"], measured["HbA1c"], 1)
r_value = np.corrcoef(measured["Glucose"], measured["HbA1c"])[0, 1]
print("HbA1c = {:.3f} + {:.4f} x Glucose".format(intercept, slope))
print("r =", round(r_value, 3), " r squared =", round(r_value ** 2, 3))

results["regression"] = {"slope": float(slope), "intercept": float(intercept),
                         "r": float(r_value), "r2": float(r_value ** 2),
                         "n": int(len(measured))}
top_pairs = corr.where(np.triu(np.ones(corr.shape), k=1) == 1).stack()
top_pairs = top_pairs.dropna().sort_values()
results["corr_top_pos"] = {
    "{} / {}".format(a, b): float(v) for (a, b), v in top_pairs.tail(5).items()
}
results["corr_top_neg"] = {
    "{} / {}".format(a, b): float(v) for (a, b), v in top_pairs.head(4).items()
}
results["corr_los"] = {
    key: float(v) for key, v in
    corr["LengthOfStay"].drop("LengthOfStay").sort_values().items()
}

## 4. Confirmatory data analysis (CDA)

Hypothesis tests are used to check, with the data, whether what we saw in
the EDA is real or could be due to chance. The significance level is
alpha = 0.05 for every test. With a very large sample the standard deviation
of the sample is a good estimate of the population standard deviation, so
the z-tests below use the sample standard deviation.

### Test 1: one-sample z-test (oxygen saturation)

A healthy oxygen saturation is around 95% or more. Is the average oxygen
saturation of our patients different from 95%?

- H0: mean oxygen saturation = 95
- H1: mean oxygen saturation is not equal to 95 (two-tailed)

In [ ]:
spo2 = df["Oxygen Saturation"]
n = len(spo2)
x_bar = spo2.mean()
s = spo2.std()
mu_0 = 95

# z = (sample mean - claimed mean) / (s / square root of n)
z_stat = (x_bar - mu_0) / (s / np.sqrt(n))
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
z_critical = stats.norm.ppf(1 - ALPHA / 2)

print("n =", n, " mean =", round(x_bar, 3), " std =", round(s, 3))
print("z =", round(z_stat, 3), " critical value = +-", round(z_critical, 3))
print("p-value =", p_value)
print("Reject H0" if p_value < ALPHA else "Fail to reject H0")

results["test1"] = {"n": int(n), "mean": float(x_bar), "std": float(s),
                    "mu0": mu_0, "z": float(z_stat), "p": float(p_value),
                    "crit": float(z_critical)}

### Test 2: one-sample t-test (HbA1c of diabetic patients)

An HbA1c of 6.5% or more is the usual cut-off for diabetes. Is the mean
HbA1c of the patients labelled Diabetes above 6.5?

- H0: mean HbA1c = 6.5
- H1: mean HbA1c > 6.5 (right-tailed)

In [ ]:
hba1c_diabetes = df.loc[df["Medical Condition"] == "Diabetes", "HbA1c"]
n = len(hba1c_diabetes)

# The population standard deviation is unknown, so a t-test is used.
t_stat, p_value = stats.ttest_1samp(hba1c_diabetes, 6.5,
                                    alternative="greater")
t_critical = stats.t.ppf(1 - ALPHA, n - 1)

# 95% confidence interval for the mean HbA1c of diabetic patients.
ci_low, ci_high = stats.t.interval(0.95, n - 1,
                                   loc=hba1c_diabetes.mean(),
                                   scale=stats.sem(hba1c_diabetes))

print("n =", n, " mean =", round(hba1c_diabetes.mean(), 3))
print("t =", round(t_stat, 3), " critical value =", round(t_critical, 3))
print("p-value =", p_value)
print("95% CI:", round(ci_low, 3), "to", round(ci_high, 3))
print("Reject H0" if p_value < ALPHA else "Fail to reject H0")

results["test2"] = {"n": int(n), "mean": float(hba1c_diabetes.mean()),
                    "t": float(t_stat), "p": float(p_value),
                    "crit": float(t_critical), "ci": [float(ci_low),
                                                      float(ci_high)]}

### Test 3: two-sample z-test (BMI of males and females)

Is the average BMI of male patients different from that of female patients?
Both groups have more than 10,000 patients, so a z-test is suitable.

- H0: mean BMI of males = mean BMI of females
- H1: the two means are different (two-tailed)

In [ ]:
known_gender = df[df["Gender"] != "Unknown"]
male_bmi = known_gender.loc[known_gender["Gender"] == "Male", "BMI"]
female_bmi = known_gender.loc[known_gender["Gender"] == "Female", "BMI"]

# z = (mean1 - mean2) / square root of (s1^2/n1 + s2^2/n2)
standard_error = np.sqrt(male_bmi.var() / len(male_bmi)
                         + female_bmi.var() / len(female_bmi))
z_stat = (male_bmi.mean() - female_bmi.mean()) / standard_error
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print("Male mean =", round(male_bmi.mean(), 3), "n =", len(male_bmi))
print("Female mean =", round(female_bmi.mean(), 3), "n =", len(female_bmi))
print("z =", round(z_stat, 3), " p-value =", round(p_value, 4))
print("Reject H0" if p_value < ALPHA else "Fail to reject H0")

results["test3"] = {"n_male": int(len(male_bmi)),
                    "n_female": int(len(female_bmi)),
                    "mean_male": float(male_bmi.mean()),
                    "mean_female": float(female_bmi.mean()),
                    "z": float(z_stat), "p": float(p_value)}

### Test 4: two-sample t-test (smokers and non-smokers, length of stay)

First we compare all smokers with all non-smokers. Then we repeat it only
for Hypertension patients, to see whether the first result is really about
smoking or about the type of condition.

- H0: mean length of stay of smokers = mean length of stay of non-smokers
- H1: the two means are different (two-tailed)

Welch's version is used (`equal_var=False`) because the two groups do not
have the same variance.

In [ ]:
# (a) All patients.
smoker_stay = df.loc[df["Smoking"] == 1, "LengthOfStay"]
non_smoker_stay = df.loc[df["Smoking"] == 0, "LengthOfStay"]
t_all, p_all = stats.ttest_ind(smoker_stay, non_smoker_stay,
                               equal_var=False)
print("All patients: smokers", round(smoker_stay.mean(), 3),
      "non-smokers", round(non_smoker_stay.mean(), 3))
print("t =", round(t_all, 3), " p-value =", p_all)

# (b) Only Hypertension patients.
hyper = df[df["Medical Condition"] == "Hypertension"]
smoker_hyper = hyper.loc[hyper["Smoking"] == 1, "LengthOfStay"]
non_smoker_hyper = hyper.loc[hyper["Smoking"] == 0, "LengthOfStay"]
t_hyper, p_hyper = stats.ttest_ind(smoker_hyper, non_smoker_hyper,
                                   equal_var=False)
print("Hypertension: smokers", round(smoker_hyper.mean(), 3),
      "non-smokers", round(non_smoker_hyper.mean(), 3))
print("t =", round(t_hyper, 3), " p-value =", round(p_hyper, 4))
print("Reject H0" if p_hyper < ALPHA else "Fail to reject H0")

results["test4"] = {
    "all": {"n_smoker": int(len(smoker_stay)),
            "n_non": int(len(non_smoker_stay)),
            "mean_smoker": float(smoker_stay.mean()),
            "mean_non": float(non_smoker_stay.mean()),
            "t": float(t_all), "p": float(p_all)},
    "hyper": {"n_smoker": int(len(smoker_hyper)),
              "n_non": int(len(non_smoker_hyper)),
              "mean_smoker": float(smoker_hyper.mean()),
              "mean_non": float(non_smoker_hyper.mean()),
              "t": float(t_hyper), "p": float(p_hyper)},
}

### Test 5: two-proportion z-test (smoking in Hypertension and Healthy)

Is the proportion of smokers among Hypertension patients different from the
proportion among Healthy people?

- H0: p1 = p2
- H1: p1 is not equal to p2 (two-tailed)

In [ ]:
smokers_hyper = int(hyper["Smoking"].sum())
n_hyper = len(hyper)
healthy = df[df["Medical Condition"] == "Healthy"]
smokers_healthy = int(healthy["Smoking"].sum())
n_healthy = len(healthy)

p_hat_1 = smokers_hyper / n_hyper
p_hat_2 = smokers_healthy / n_healthy

# Pooled proportion under H0.
p_pooled = (smokers_hyper + smokers_healthy) / (n_hyper + n_healthy)
standard_error = np.sqrt(p_pooled * (1 - p_pooled)
                         * (1 / n_hyper + 1 / n_healthy))
z_stat = (p_hat_1 - p_hat_2) / standard_error
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

print("Hypertension:", round(p_hat_1, 3), " Healthy:", round(p_hat_2, 3))
print("z =", round(z_stat, 3), " p-value =", p_value)
print("Reject H0" if p_value < ALPHA else "Fail to reject H0")

results["test5"] = {"p1": float(p_hat_1), "p2": float(p_hat_2),
                    "n1": int(n_hyper), "n2": int(n_healthy),
                    "z": float(z_stat), "p": float(p_value)}

In [ ]:
# Summary of all five tests.
summary_tests = pd.DataFrame({
    "Test": ["1 One-sample z", "2 One-sample t", "3 Two-sample z",
             "4b Two-sample t (Hypertension)", "5 Two-proportion z"],
    "Statistic": [results["test1"]["z"], results["test2"]["t"],
                  results["test3"]["z"], results["test4"]["hyper"]["t"],
                  results["test5"]["z"]],
    "p-value": [results["test1"]["p"], results["test2"]["p"],
                results["test3"]["p"], results["test4"]["hyper"]["p"],
                results["test5"]["p"]],
})
summary_tests["Decision"] = np.where(summary_tests["p-value"] < ALPHA,
                                     "Reject H0", "Fail to reject H0")
summary_tests.round(4)

## 5. Findings and conclusions

- The condition decides how long a patient stays: about
  1.5 days for Healthy, 3.5 to 4 days for
  Arthritis, Obesity, Asthma and Hypertension, 5.5
  for Diabetes and 13.0 for Cancer.
- Test 1: the mean oxygen saturation (94.57%) is statistically
  different from 95% (z = -20.35), but the gap is only
  0.43 points, so it is not practically important.
- Test 2: the mean HbA1c of diabetic patients is 8.01%, clearly
  above 6.5% (t = 101.6, p < 0.001).
- Test 3: male and female BMI are not different (z = -0.33,
  p = 0.74), so we fail to reject H0.
- Test 4: smokers stay longer overall (4.82 vs
  4.24 days), but within Hypertension patients there is no
  difference (p = 0.79). Smoking is more common in the long-stay
  conditions, so the overall difference comes from the condition.
- Test 5: smoking is much more common in Hypertension patients
  (41.1%) than in Healthy people (10.7%),
  z = 30.1.
- Glucose and HbA1c rise together (r = 0.62).

**Limitations:** filling with the mean makes the data slightly less spread
out, and the dataset looks artificial, so the results describe this dataset
only and do not prove cause and effect.

In [ ]:
# Save the numbers that are quoted in the report.
with open("results.json", "w") as result_file:
    json.dump(results, result_file, indent=2)
print("Saved results.json")